# Task 2 — Validation & Demo 
**(Cleaning → Standardisation → Enrichment → Validation)**

This notebook demonstrates that the Task 2 preprocessing pipeline works end-to-end on HDFS using PySpark. It imports the single-class modules created for Task 2 and shows evidence for each stage.

**Stages**  
1. Load a raw slice from HDFS
2. Cleaning & Standardisation
3. Enrichment (per-capita metrics + ISO reference join)
4. Validation (split into valid/rejects + summary)
5. Write processed / rejects / summary to HDFS and verify

In [1]:
import pyspark
from pyspark.sql import SparkSession
print("PySpark version:", pyspark.__version__)


PySpark version: 3.5.1


In [2]:
DATE = None  # e.g., "2025-08-22" to demo a single day; or None for all
RAW_PATH = "hdfs://localhost:9000/user/student/covid19/raw"
PROCESSED_PATH = "hdfs://localhost:9000/user/student/covid19/processed"
REJECTS_PATH = "hdfs://localhost:9000/user/student/covid19/rejects"
SUMMARY_PATH = "hdfs://localhost:9000/user/student/covid19/validation_summary"
ISO_REF_PATH = "hdfs://localhost:9000/user/student/covid19/ref/iso_ref.csv"


**Import Task 2 Classes**

In [3]:
import os, sys, pathlib

def _try_add(p):
    if os.path.isdir(p) and p not in sys.path:
        sys.path.append(p)

cwd = os.getcwd()
_try_add(os.path.join(cwd, "classes"))
_try_add(os.path.join(cwd, "..", "classes"))
_try_add(str(pathlib.Path(cwd).resolve().parent / "classes"))

from task2_clean_and_standardise import CleaningAndStandardisation
from task2_enrichment import EnrichmentTransformer
from task2_validation import ValidationChecks


In [4]:
spark = (SparkSession.builder
         .appName("task2_demo_validate")
         .config("spark.hadoop.fs.defaultFS", "hdfs://localhost:9000")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")


25/08/27 18:59:21 WARN Utils: Your hostname, ZGMF-X666S.localdomain resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/08/27 18:59:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/27 18:59:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/08/27 18:59:23 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


**Load raw data from HDFS**

In [5]:
from pyspark.sql.functions import col, date_format

df_raw = spark.read.parquet(RAW_PATH)
if DATE and "event_date" in df_raw.columns:
    df_raw = df_raw.where(date_format(col("event_date"), "yyyy-MM-dd") == DATE)

print("Raw rows:", df_raw.count())
df_raw.printSchema()
df_raw.show(5, truncate=False)


Raw rows: 109725
root
 |-- country: string (nullable = true)
 |-- continent: string (nullable = true)
 |-- cases: long (nullable = true)
 |-- todayCases: long (nullable = true)
 |-- deaths: long (nullable = true)
 |-- todayDeaths: long (nullable = true)
 |-- recovered: long (nullable = true)
 |-- active: long (nullable = true)
 |-- critical: long (nullable = true)
 |-- tests: long (nullable = true)
 |-- population: long (nullable = true)
 |-- casesPerOneMillion: double (nullable = true)
 |-- deathsPerOneMillion: double (nullable = true)
 |-- updated: long (nullable = true)
 |-- event_ts: timestamp (nullable = true)
 |-- event_date: date (nullable = true)

+-----------+-------------+--------+----------+------+-----------+---------+--------+--------+---------+----------+------------------+-------------------+-------------+-------------------+----------+
|country    |continent    |cases   |todayCases|deaths|todayDeaths|recovered|active  |critical|tests    |population|casesPerOneMillion|de

**Cleaning and Standardisation**

In [6]:
# ----- BEFORE vs AFTER summary for demo -----
from pyspark.sql import functions as F
import pandas as pd

def _safe_sum(df, colname):
    return (df.select(F.sum(colname)).first()[0]
            if colname in df.columns else None)

def _safe_count_distinct(df, colname):
    return (df.select(F.col(colname)).na.drop().distinct().count()
            if colname in df.columns else None)

def _fmt(x):
    if x is None: return "—"
    try: return f"{int(x):,}"
    except: return str(x)

# 1) Run cleaning/standardisation
cleaner = CleaningAndStandardisation()
df_clean = cleaner.run(df_raw).persist()
print("After clean/std rows:", df_clean.count())

# 2) KPIs before vs after
kpi_rows = [
    ["Rows",
     _fmt(df_raw.count()),
     _fmt(df_clean.count())],
    ["Countries (distinct)",
     _fmt(_safe_count_distinct(df_raw, "country")),
     _fmt(_safe_count_distinct(df_clean, "country"))],
    ["Total cases",
     _fmt(_safe_sum(df_raw, "cases")),
     _fmt(_safe_sum(df_clean, "cases"))],
    ["Total deaths",
     _fmt(_safe_sum(df_raw, "deaths")),
     _fmt(_safe_sum(df_clean, "deaths"))],
    ["Total tests",
     _fmt(_safe_sum(df_raw, "tests")),
     _fmt(_safe_sum(df_clean, "tests"))],
]
display(pd.DataFrame(kpi_rows, columns=["Metric", "Before (raw)", "After (processed)"]))

# 3) Nulls removed for key columns (before vs after)
key_cols = [c for c in ["country","cases","deaths","tests","population","updated","event_date"] if c in df_raw.columns or c in df_clean.columns]
null_rows = []
for c in key_cols:
    before_nulls = df_raw.filter(F.col(c).isNull()).count() if c in df_raw.columns else None
    after_nulls  = df_clean.filter(F.col(c).isNull()).count() if c in df_clean.columns else None
    null_rows.append([c, _fmt(before_nulls), _fmt(after_nulls)])
display(pd.DataFrame(null_rows, columns=["Column", "Nulls (before)", "Nulls (after)"]))

# 4) Duplicate reduction check on a stable key (country + updated) if available
dup_table = []
if {"country","updated"}.issubset(set(df_raw.columns)):
    raw_dups = (df_raw
                .groupBy("country","updated")
                .count().filter("count > 1").select(F.sum("count") - F.count("*")).first()[0])
    raw_dups = raw_dups or 0
    dup_table.append(["Duplicate rows on (country, updated) – BEFORE", _fmt(raw_dups)])
if {"country","updated"}.issubset(set(df_clean.columns)):
    clean_dups = (df_clean
                  .groupBy("country","updated")
                  .count().filter("count > 1").select(F.sum("count") - F.count("*")).first()[0])
    clean_dups = clean_dups or 0
    dup_table.append(["Duplicate rows on (country, updated) – AFTER", _fmt(clean_dups)])
if dup_table:
    display(pd.DataFrame(dup_table, columns=["Metric", "Value"]))

# 5) Example of standardisation: country name fixes (top 10 changed)
if "country" in df_raw.columns and "country" in df_clean.columns:
    before_names = (df_raw
                    .select(F.lower(F.col("country")).alias("raw_name"))
                    .na.drop().distinct())
    after_names  = (df_clean
                    .select(F.col("country").alias("clean_name"),
                            F.lower(F.col("country")).alias("raw_name_norm"))
                    .na.drop().distinct())
    country_fix_examples = (before_names
                            .join(after_names, before_names.raw_name == after_names.raw_name_norm, "left")
                            .select(F.col("raw_name").alias("raw_country"),
                                    F.col("clean_name"))
                            .orderBy("raw_country")
                            .limit(10))
    print("Sample country standardisation (first 10):")
    country_fix_examples.show(truncate=False)

# 6) Show a compact sample of BEFORE vs AFTER rows side-by-side (if key exists)
if {"country","updated"}.issubset(set(df_raw.columns)) and {"country","updated"}.issubset(set(df_clean.columns)):
    cols = ["country","cases","deaths","tests","population","updated"]
    before_small = df_raw.select([c for c in cols if c in df_raw.columns]).limit(5)
    after_small  = df_clean.select([c for c in cols if c in df_clean.columns]).limit(5)
    print("\nBEFORE (sample):")
    before_small.show(truncate=False)
    print("AFTER (sample):")
    after_small.show(truncate=False)

# 7) Your original “after” sample (kept for continuity)
sel_cols = [c for c in ["country","iso2","iso3","cases","deaths","tests","population","event_date","updated_ts"] if c in df_clean.columns]
if sel_cols:
    df_clean.select(*sel_cols).show(5, truncate=False)


25/08/27 18:59:33 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


After clean/std rows: 231


,Metric,Before (raw),After (processed)
0,Rows,"109,725",231
1,Countries (distinct),231,231
2,Total cases,"334,759,613,709","704,757,817"
3,Total deaths,"3,330,099,602","7,010,724"
4,Total tests,"3,337,592,268,287","7,026,510,052"


,Column,Nulls (before),Nulls (after)
0,country,0,0
1,cases,0,0
2,deaths,0,0
3,tests,0,0
4,population,0,0
5,updated,0,0
6,event_date,0,0


,Metric,Value
0,"Duplicate rows on (country, updated) – BEFORE",0
1,"Duplicate rows on (country, updated) – AFTER",0


Sample country standardisation (first 10):


+-------------------+-------------------+
|raw_country        |clean_name         |
+-------------------+-------------------+
|afghanistan        |Afghanistan        |
|albania            |Albania            |
|algeria            |Algeria            |
|andorra            |Andorra            |
|angola             |Angola             |
|anguilla           |Anguilla           |
|antigua and barbuda|Antigua And Barbuda|
|argentina          |Argentina          |
|armenia            |Armenia            |
|aruba              |Aruba              |
+-------------------+-------------------+


BEFORE (sample):
+-----------+--------+------+---------+----------+-------------+
|country    |cases   |deaths|tests    |population|updated      |
+-----------+--------+------+---------+----------+-------------+
|St. Barth  |5507    |6     |78646    |9945      |1756261723712|
|Japan      |33803625|74696 |100415017|125584838 |1756261723712|
|Thailand   |4770198 |34587 |17273513 |70078203  |1756261723712|
|Sw

**Enrichment**

In [7]:
# ======================
# ENRICHMENT 
# ======================
from pyspark.sql import functions as F

# 0) Run enrichment
enricher = EnrichmentTransformer(spark, iso_ref_path=ISO_REF_PATH)
df_enriched = enricher.run(df_clean).persist()

# 1) Row-count check
cnt_before = df_clean.count()
cnt_after  = df_enriched.count()
print(f"After enrichment rows: {cnt_after}")
print(f"[CHECK] Row count unchanged? before={cnt_before:,} after={cnt_after:,} -> {'OK' if cnt_before == cnt_after else 'MISMATCH'}")

# 2) Show only the enriched columns you care about
wanted_new_cols = ["cases_per_million", "deaths_per_million", "tests_per_thousand", "ingest_time", "source_system"]
present_new_cols = [c for c in wanted_new_cols if c in df_enriched.columns]
print("[CHECK] New columns present:", present_new_cols)
print()

# 3) BEFORE sample (cleaned)
before_cols = [c for c in ["country","cases","deaths","tests","population","event_date"] if c in df_clean.columns]
print("[BEFORE] Cleaned sample:")
df_clean.select(*before_cols).orderBy("country").limit(10).show(truncate=False)

# 4) AFTER sample (enriched) — same base cols + enriched cols
after_cols = before_cols + [c for c in wanted_new_cols if c in df_enriched.columns]
print("[AFTER]  Enriched sample:")
df_enriched.select(*after_cols).orderBy("country").limit(10).show(truncate=False)

# 5) OPTIONAL: a second AFTER view matching your final table layout
alt_after_cols = [c for c in [
    "country","cases","deaths","tests","population",
    "cases_per_million","deaths_per_million","tests_per_thousand",
    "event_date","ingest_time","source_system"
] if c in df_enriched.columns]
if alt_after_cols:
    df_enriched.select(*alt_after_cols).orderBy("country").limit(10).show(truncate=False)


After enrichment rows: 231
[CHECK] Row count unchanged? before=231 after=231 -> OK
[CHECK] New columns present: ['cases_per_million', 'deaths_per_million', 'tests_per_thousand', 'ingest_time', 'source_system']

[BEFORE] Cleaned sample:
+-------------------+--------+------+--------+----------+----------+
|country            |cases   |deaths|tests   |population|event_date|
+-------------------+--------+------+--------+----------+----------+
|Afghanistan        |234203  |7996  |1390754 |40754388  |2025-08-27|
|Albania            |334863  |3605  |1941032 |2866374   |2025-08-27|
|Algeria            |272024  |6881  |230991  |45350148  |2025-08-27|
|Andorra            |48015   |165   |249839  |77463     |2025-08-27|
|Angola             |107351  |1937  |1499833 |35027343  |2025-08-27|
|Anguilla           |3904    |12    |51382   |15230     |2025-08-27|
|Antigua And Barbuda|9107    |146   |18901   |99509     |2025-08-27|
|Argentina          |10128864|130841|35716075|46010234  |2025-08-27|
|Arme

**Validation**

In [8]:
# =========================
# VALIDATION 
# =========================
from pyspark.sql import functions as F
import pandas as pd

validator = ValidationChecks()
valid_df, invalid_df, summary_df = validator.split(df_enriched)

total = df_enriched.count()
n_valid = valid_df.count()
n_invalid = invalid_df.count()

print(f"Valid rows: {n_valid:,}  Invalid rows: {n_invalid:,}  (Total: {total:,})\n")

# 1) State the rules
RULES = {
    "population_positive"  : "population > 0",
    "deaths_le_cases"      : "deaths <= cases",
    "tests_ge_cases"       : "tests >= cases",
    "cases_le_population"  : "cases <= population",
    "deaths_le_population" : "deaths <= population",
}
print("Validation rules:")
for k, v in RULES.items():
    print(f"  • {k}: {v}")
print()

# 2) Pretty summary
sum_row = summary_df.first().asDict()
rows = []
for key, val in sum_row.items():
    if key.startswith("violations__"):
        name = key.split("__", 1)[1]
        viol = int(val or 0)
        rows.append([name, viol, total - viol, f"{(viol/total):.1%}"])
order = ["population_positive","deaths_le_cases","tests_ge_cases","cases_le_population","deaths_le_population"]
rows = sorted(rows, key=lambda r: order.index(r[0]) if r[0] in order else 999)

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 160)
summary_table = pd.DataFrame(rows, columns=["rule", "violations", "passes", "violation_rate"])
print("Summary of validation results:")
print(summary_table.to_string(index=False))
print()

# 3) Invalid row sample (simplified)
flags = (invalid_df
         .withColumn("viol_population_positive",  (F.col("population") <= 0))
         .withColumn("viol_deaths_le_cases",      (F.col("deaths") >  F.col("cases")))
         .withColumn("viol_tests_ge_cases",       (F.col("tests")  <  F.col("cases")))
         .withColumn("viol_cases_le_population",  (F.col("cases")  >  F.col("population")))
         .withColumn("viol_deaths_le_population", (F.col("deaths") >  F.col("population")))
        )

# Keep only essential columns
nice_cols = [c for c in [
    "country","continent","cases","deaths","tests","population",
    "cases_per_million","deaths_per_million","tests_per_thousand"
] if c in flags.columns]

flag_cols = [c for c in [
    "viol_population_positive","viol_deaths_le_cases","viol_tests_ge_cases",
    "viol_cases_le_population","viol_deaths_le_population"
] if c in flags.columns]

sample_pdf = (flags
              .select(*nice_cols, *flag_cols)
              .orderBy(F.desc("cases"))
              .limit(10)
              .toPandas())

# Format numbers
def _fmt(x):
    if isinstance(x, (int, float)):
        return f"{x:,.2f}" if not float(x).is_integer() else f"{int(x):,}"
    return x

for col in ["cases","deaths","tests","population"]:
    if col in sample_pdf.columns:
        sample_pdf[col] = sample_pdf[col].map(_fmt)

for col in ["cases_per_million","deaths_per_million","tests_per_thousand"]:
    if col in sample_pdf.columns:
        sample_pdf[col] = sample_pdf[col].map(lambda v: f"{v:,.2f}" if pd.notnull(v) else v)

print("Invalid rows (sample, with which rules failed):")
print(sample_pdf.to_string(index=False))


Valid rows: 210  Invalid rows: 21  (Total: 231)

Validation rules:
  • population_positive: population > 0
  • deaths_le_cases: deaths <= cases
  • tests_ge_cases: tests >= cases
  • cases_le_population: cases <= population
  • deaths_le_population: deaths <= population

Summary of validation results:
                rule  violations  passes violation_rate
 population_positive           2     229           0.9%
     deaths_le_cases           0     231           0.0%
      tests_ge_cases          21     210           9.1%
 cases_le_population           2     229           0.9%
deaths_le_population           2     229           0.9%

Invalid rows (sample, with which rules failed):
         country         continent      cases deaths      tests population cases_per_million deaths_per_million tests_per_thousand  viol_population_positive  viol_deaths_le_cases  viol_tests_ge_cases  viol_cases_le_population  viol_deaths_le_population
        S. Korea              Asia 34,571,879 35,935 15,804

In [9]:
w_valid = valid_df
if "event_date" in w_valid.columns:
    (w_valid.write.mode("append").partitionBy("event_date").parquet(PROCESSED_PATH))
else:
    (w_valid.write.mode("append").parquet(PROCESSED_PATH))

if invalid_df.limit(1).count() > 0:
    w_invalid = invalid_df
    if "event_date" in w_invalid.columns:
        (w_invalid.write.mode("append").partitionBy("event_date").parquet(REJECTS_PATH))
    else:
        (w_invalid.write.mode("append").parquet(REJECTS_PATH))

(summary_df.write.mode("append").parquet(SUMMARY_PATH))
print("Write completed.")


25/08/27 19:00:03 WARN MemoryManager: Total allocation exceeds 50.00% (477,364,224 bytes) of heap memory
Scaling row group sizes to 88.92% for 4 writers
25/08/27 19:00:03 WARN MemoryManager: Total allocation exceeds 50.00% (477,364,224 bytes) of heap memory
Scaling row group sizes to 71.13% for 5 writers
25/08/27 19:00:03 WARN MemoryManager: Total allocation exceeds 50.00% (477,364,224 bytes) of heap memory
Scaling row group sizes to 59.28% for 6 writers
25/08/27 19:00:03 WARN MemoryManager: Total allocation exceeds 50.00% (477,364,224 bytes) of heap memory
Scaling row group sizes to 50.81% for 7 writers
25/08/27 19:00:03 WARN MemoryManager: Total allocation exceeds 50.00% (477,364,224 bytes) of heap memory
Scaling row group sizes to 44.46% for 8 writers
25/08/27 19:00:03 WARN MemoryManager: Total allocation exceeds 50.00% (477,364,224 bytes) of heap memory
Scaling row group sizes to 39.52% for 9 writers
25/08/27 19:00:03 WARN MemoryManager: Total allocation exceeds 50.00% (477,364,224

Write completed.


In [10]:
p_valid = spark.read.parquet(PROCESSED_PATH)
print("Processed count:", p_valid.count())

p_summary = spark.read.parquet(SUMMARY_PATH)
print("Summary count:", p_summary.count())
p_summary.show(5, truncate=False)

try:
    p_rejects = spark.read.parquet(REJECTS_PATH)
    print("Rejects count:", p_rejects.count())
except Exception:
    print("Rejects path empty or not created.")


Processed count: 420
Summary count: 2
+-------------------------------+---------------------------+--------------------------+-------------------------------+--------------------------------+----------+
|violations__population_positive|violations__deaths_le_cases|violations__tests_ge_cases|violations__cases_le_population|violations__deaths_le_population|total_rows|
+-------------------------------+---------------------------+--------------------------+-------------------------------+--------------------------------+----------+
|2                              |0                          |21                        |2                              |2                               |231       |
|2                              |0                          |21                        |2                              |2                               |231       |
+-------------------------------+---------------------------+--------------------------+-------------------------------+-----------------

In [11]:
spark.stop()